# 2-stage playground — inference + evaluation (DIVA-HisDB & U-DIADS-TL, merged)

Loads the trained detector for `(BACKBONE, FAMILY, SUBSET)`, runs stage-1 boxes +
stage-2 crop segmentation on a page, visualises, then scores the full split with the
official per-family tool. Replaces the old `99_playground_cb55` + `99_playground_u_diads`.

## Parameters

In [ ]:
# ==================== PARAMETERS (identical block in every 02_2stage notebook) ====================
# Switch the whole notebook by editing these three. BACKBONE is any key from
# train_detector.BACKBONES; FAMILY/SUBSET pick the dataset subset.
BACKBONE = "dinov2"      # resnet34 resnet50 convnext convnext_dinov3
                          # vit_small_patch16_224.augreg_in21k vit_small_patch16_dinov3
                          # dinov2 dinov2_reg am-radio
FAMILY   = "diva"        # "diva" | "udiads"
SUBSET   = "CB55"        # diva: CB55 CS18 CS863   |   udiads: Latin14396 Latin2 Syr341
IMGSZ    = 1280           # detector inference/training size (multiple of 32)
CONF     = 0.25           # detector confidence threshold
CROP_IMGSZ = 640          # stage-2 crop-segmenter size

## Resolve paths for the chosen backbone / subset

In [ ]:
import sys, torch
from pathlib import Path
sys.path.insert(0, ".")
import train_detector

assert BACKBONE in train_detector.BACKBONES, f"unknown BACKBONE {BACKBONE!r}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# (FAMILY, SUBSET) -> detection data config + trained-detector checkpoint path.
DATA_CFG = Path(f"configs/data/{('diva_'+SUBSET.lower()) if FAMILY=='diva' else ('udiads_'+SUBSET.lower())}_detect.yaml")
FAM_DIR = "diva-hisdb" if FAMILY == "diva" else "u-diads-tl"
RUN = f"yolov8_{BACKBONE}_{FAMILY}_{SUBSET}"
DET_WEIGHTS = Path(f"../../80_models/02_2stage/{FAM_DIR}/detection/{RUN}/weights/best.pt")
print("device      :", DEVICE)
print("data config :", DATA_CFG, "(exists)" if DATA_CFG.exists() else "(MISSING)")
print("detector    :", DET_WEIGHTS, "(exists)" if DET_WEIGHTS.exists() else "(train it first)")
# Absolute data YAML (fixes Ultralytics resolving `path:` vs datasets_dir).
DATA = train_detector.resolve_data(DATA_CFG) if DATA_CFG.exists() else str(DATA_CFG)


## Load detector + a crop-segmenter checkpoint

In [ ]:
from ultralytics import YOLO
# Detector: reload the trained checkpoint (backbone classes are registered on import
# of train_detector, so hier_encoder SFP checkpoints unpickle cleanly).
train_detector.build_model  # ensure registration side-effects have run
detector = YOLO(str(DET_WEIGHTS)) if DET_WEIGHTS.exists() else None

# Crop segmenter: pick a trained stage-2 seg checkpoint (see train_crops.ipynb).
SEG_WEIGHTS = Path(f"../../80_models/02_2stage/{FAM_DIR}/crop_seg/seg_{BACKBONE}_{FAMILY}_{SUBSET}/best.pth")
print("seg weights :", SEG_WEIGHTS, "(exists)" if SEG_WEIGHTS.exists() else "(train it first)")

## Single page: detect → crop-segment → visualise

In [ ]:
import random, cv2, numpy as np, matplotlib.pyplot as plt
import yaml as _yaml
d = _yaml.safe_load(open(DATA))
img_dir = (Path(d["path"]) / d["test"]).resolve()
stems = sorted(p.stem for p in img_dir.glob("*.*")) if img_dir.exists() else []
print(len(stems), "test images in", img_dir)

if detector and stems:
    stem = random.choice(stems)
    img_path = next(img_dir.glob(stem + ".*"))
    res = detector.predict(str(img_path), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    vis = res.plot()
    plt.figure(figsize=(14, 10)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"{BACKBONE} · {FAMILY}/{SUBSET} · {stem} · {len(res.boxes)} lines"); plt.axis("off"); plt.show()
else:
    print("Train the detector first (train_detection_colab.ipynb / run_train_all.sh).")

## Full-split evaluation (2-stage → PAGE-XML → official score)

Uses `evaluate.py`'s CLI: `predict` runs detector+segmenter over the split into PAGE-XML,
then the per-family `score-*` subcommand computes the official metrics (DIVA Java tool /
U-DIADS instance metric).

In [ ]:
# Two-stage predict -> PAGE-XML, then score. Fill in test dirs from the data config.
XML_OUT = Path(f"/tmp/eval_{RUN}/pred_xml"); XML_OUT.mkdir(parents=True, exist_ok=True)
cmd = f"""python evaluate.py predict \
  --dataset-family {FAMILY} \
  --yolo-weights {DET_WEIGHTS} \
  --segm-weights {SEG_WEIGHTS} \
  --test-img-dir {img_dir} \
  --out-xml-dir {XML_OUT}"""
print(cmd)
# !{cmd}    # uncomment to run (needs trained detector + segmenter)